# Heston calibration

The aim of this notebook is to illustrate the calibration of the Heston model on a CAC40 option chain,
and to compare the implied volatility surface reconstructed by the model with the market one.

**Pipeline :**
1. Load the market data : EURIBOR 6M zero-coupon rates and CAC40 option prices (12/02/2025).
2. Build the implied forward curve from the Call / Put parity (`ForwardCurve`).
3. Build the market implied volatility surface (`BlackScholes` + `ImpliedVolatilitySurface`).
4. Calibrate the Heston model on this surface (`HestonModel`).
5. Search the global minimum of the calibration loss (`Calibrator` : Latin Hypercube, Differential Evolution, Tikhonov).
6. Reconstruct the Heston implied volatility surface and compare it with the market.
7. Check the static no-arbitrage conditions on the interpolated market and Heston surfaces.
8. Calibrate a time-dependent Heston model (piecewise constant parameters).
9. Test the calibrations for overfitting (out-of-sample errors, stability under noise).

In [4]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.interpolate import PchipInterpolator

from forward_curve import ForwardCurve
from Black_Scholes import BlackScholes
from ImpliedVolatilitySurface import ImpliedVolatilitySurface
from Heston import HestonModel
from Heston import TimeDependentHestonModel
from error import CalibrationLoss
from optimizer import Calibrator
from scipy.stats import norm

## 1. Market data

### 1.1 Data loading

In [ ]:
eq_underlying = "CAC40"
rate_underlying = "EURIBOR6M"
S0 = 8042.19
as_of_date = datetime.date(2025, 2, 12)

df_rates = pd.read_csv("../Data/EURIBOR6M_ZCRates_12022025.csv")
df_options = pd.read_csv("../Data/CAC40_MarketOptions_12022025.csv")

### 1.2 Zero-coupon rate curve

The zero-coupon rates are interpolated with a PCHIP interpolator (monotone, no overshoot), extrapolated beyond the last pillar.

In [ ]:
if not df_rates['Expiry'].is_monotonic_increasing:
    raise ValueError("df_rates['Expiry'] is not monotonically increasing. Sort the data.")
zc_rate = PchipInterpolator(df_rates['Expiry'], df_rates['ZCRate'], extrapolate=True)

T_fine = np.linspace(df_rates['Expiry'].min(), df_rates['Expiry'].max(), 200)
plt.figure(figsize=(9, 4.5))
plt.plot(T_fine, 100 * zc_rate(T_fine), color='navy', linewidth=2, label='PCHIP interpolation')
plt.scatter(df_rates['Expiry'], 100 * df_rates['ZCRate'], color='red', zorder=5, label=f'Market data ({rate_underlying})')
plt.title(f"{rate_underlying} zero-coupon curve")
plt.xlabel('Maturity T (years)')
plt.ylabel('Zero-coupon rate (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

### 1.3 Option chain

The listed expiries are given as `Month-Year` : options expire on the third Friday of the month,
which gives the time to expiry in years (ACT/365).

In [ ]:
def third_friday(month, year):
    """
    Returns the third friday of month/year.
    """
    result = datetime.date(year, month, 15)
    return result.replace(day=15 + (4 - result.weekday()) % 7)

months_nb = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, "July": 7,
             "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}

def year_fraction(label):
    month, year = label.split("-")
    return (third_friday(months_nb[month], int(year)) - as_of_date).days / 365

df_options['Expiry'] = df_options['Expiry'].apply(year_fraction)
print(f"{eq_underlying} market options :")
df_options

In [ ]:
def collect_data(option_data):
    """
    Collects options market data and stores it in dictionaries indexed by expiry.
    """
    # Sanity checks
    if option_data is None or option_data.empty:
        raise ValueError("option_data cannot be None or empty.")
    if not all(col in option_data.columns for col in ['Expiry', 'Strike', 'Call', 'Put']):
        raise ValueError("option_data must contain columns: 'Expiry', 'Strike', 'Call', 'Put'")
    if (option_data['Call'] < 0).any() or (option_data['Put'] < 0).any():
        raise ValueError("Option prices must be non-negative.")
    if (option_data['Strike'] <= 0).any():
        raise ValueError("Strikes must be positive.")
    # Data collection
    strikes, call_prices, put_prices = dict(), dict(), dict()
    for exp in np.sort(option_data['Expiry'].unique()):
        df = option_data[option_data['Expiry'] == exp].sort_values(by='Strike')
        strikes[exp], call_prices[exp], put_prices[exp] = list(df['Strike']), list(df['Call']), list(df['Put'])
    return {"Strikes": strikes, "Call Prices": call_prices, "Put Prices": put_prices}

data = collect_data(df_options)
market_strikes = data["Strikes"]
call_prices = data["Call Prices"]
put_prices = data["Put Prices"]
listed_expiries = list(market_strikes.keys())

In [ ]:
expiry = listed_expiries[-1]
x_axis = np.linspace(market_strikes[expiry][0], market_strikes[expiry][-1], 101)
call_price_func = PchipInterpolator(market_strikes[expiry], call_prices[expiry])
put_price_func = PchipInterpolator(market_strikes[expiry], put_prices[expiry])

plt.figure(figsize=(8, 5))
plt.scatter(market_strikes[expiry], call_prices[expiry], color="black", label="Market call price")
plt.scatter(market_strikes[expiry], put_prices[expiry], color="blue", label="Market put price")
plt.plot(x_axis, call_price_func(x_axis), color="red", label="PCHIP call")
plt.plot(x_axis, put_price_func(x_axis), color="green", label="PCHIP put")
plt.title(f"{eq_underlying} market options - T = {expiry:.2f}y")
plt.xlabel("Strike")
plt.ylabel("Option price")
plt.legend(loc="upper left")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 2. Implied forward curve

For each expiry, the implied forward is the strike $K$ such that $C(K) - P(K) = 0$ (Call / Put parity).
The forwards give the implied *borrow + dividend* yield, assumed piecewise constant between listed expiries.

In [ ]:
plt.rcParams["figure.figsize"] = (8, 4.5)
forward_curve = ForwardCurve(S0, zc_rate, listed_expiries, market_strikes, call_prices, put_prices)

In [ ]:
T_grid = np.linspace(1e-3, listed_expiries[-1], 300)
plt.figure(figsize=(8, 4.5))
plt.plot(T_grid, [forward_curve.calc_forward(t) for t in T_grid], color='navy', label='Forward curve F(0,T)')
plt.scatter(listed_expiries, list(forward_curve.implied_forwards.values()), color='red', zorder=5, label='Implied forwards')
plt.axhline(S0, color='grey', linestyle=':', label='Spot')
plt.title(f"{eq_underlying} implied forward curve")
plt.xlabel("Expiry (years)")
plt.ylabel("Forward")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 3. Market implied volatility surface

Only out-of-the-money options are used (puts below the forward, calls above) since they are the most liquid.
Their Black implied volatilities are obtained by inverting the `BlackScholes` pricer (Newton-Raphson, with a Brent fallback).

In [ ]:
def collect_otm_options(listed_expiries, market_strikes, call_prices, put_prices, implied_forwards):
    """
    Filters the listed options and collects the OTM ones.
    """
    if not implied_forwards:
        raise ValueError("implied_forwards cannot be empty.")
    otm_options = dict()
    for exp in listed_expiries:
        otm_options[exp] = [put if K <= implied_forwards[exp] else call
                            for K, call, put in zip(market_strikes[exp], call_prices[exp], put_prices[exp])]
    return otm_options

otm_options = collect_otm_options(listed_expiries, market_strikes, call_prices, put_prices, forward_curve.implied_forwards)
bs_model = BlackScholes(zc_rate, forward_curve)
market_iv_surface = ImpliedVolatilitySurface(forward_curve, bs_model)
market_iv_surface.set_implied_vol_surface(market_strikes, otm_options)

In [ ]:
market_iv_surface.plot_implied_vol_surface(title=f"{eq_underlying} market implied volatility surface");

## 4. Heston calibration

$$
dS_t = \mu_t S_t\,dt + \sqrt{v_t}\,S_t\,dW^S_t, \qquad
dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW^v_t, \qquad
d\langle W^S, W^v\rangle_t = \rho\,dt
$$

The drift is taken from the implied forward curve, so the model reprices the forwards exactly.
Prices are computed with the Lewis (2001) single-integral formula.
The calibration minimises the vega-weighted price errors $(P^{Heston} - P^{mkt}) / \mathcal{V}^{mkt}$,
a first-order approximation of the implied volatility error, averaged per expiry so that each expiry weighs the same.

In [ ]:
heston = HestonModel(zc_rate, forward_curve, v0=0.04, theta=0.04, kappa=2.0, xi=0.3, rho=-0.7)
calibration = heston.calibrate(market_iv_surface, bs_model)
pd.Series(heston.params, name="Calibrated Heston parameters").to_frame()

## 5. Global calibration

The L-BFGS-B run of section 4 is a **local** search : its result depends on the starting point, and the loss has
several local minima as well as a flat valley along $(\kappa, \theta, \xi)$.
The `Calibrator` class (`optimizer.py`) searches the global minimum of the same vega-weighted loss (`CalibrationLoss`, `error.py`).
All its searches work on parameters rescaled to $[0,1]$, so that $v_0 \approx 0.02$ and $\kappa \approx 2$ weigh alike in the optimizer.

### 5.1 Latin Hypercube multi-start

200 parameter sets are drawn with a Latin Hypercube over the bounds and the loss is evaluated on each of them.
A local L-BFGS-B minimisation is then run from the 10 best points.
Several runs reaching the same point is evidence that it is the global minimum.

In [ ]:
local_params = heston.params          # section 4 result, kept for comparison
calibration_loss = CalibrationLoss(market_iv_surface, bs_model)
rmse = lambda loss_value: 100 * np.sqrt(loss_value)     # vega-weighted loss -> RMSE in vol points
params = list(HestonModel.PARAMS)

lhs_calibrator = Calibrator(heston, calibration_loss, loss_name='vega_iv_mse')
best_lhs = lhs_calibrator.run('lhs', n_samples=200, n_starts=10, seed=0)

lhs_runs = lhs_calibrator.summary()
lhs_runs["RMSE (vol pts)"] = rmse(lhs_runs["loss"])
lhs_runs.drop(columns=["loss", "objective"]).round(4)

In [ ]:
minima = lhs_calibrator.distinct_minima()
minima["RMSE (vol pts)"] = rmse(minima["objective"])
minima.drop(columns="objective").round(4)

### 5.2 Differential Evolution

Population-based global search (population initialised with a Latin Hypercube), followed by an L-BFGS-B polish.
It is independent of the multi-start and serves as a cross-check.

In [ ]:
de_calibrator = Calibrator(HestonModel(zc_rate, forward_curve), calibration_loss, loss_name='vega_iv_mse')
best_de = de_calibrator.run('de', popsize=15, maxiter=100, seed=0)

In [ ]:
comparison = pd.DataFrame({
    "Local L-BFGS-B (section 4)": {**local_params, "RMSE (vol pts)": rmse(calibration.fun), "Evaluations": calibration.nfev},
    "Latin Hypercube multi-start": {**{p: best_lhs[p] for p in params}, "RMSE (vol pts)": rmse(best_lhs["loss"]),
                                    "Evaluations": lhs_calibrator.n_evals},
    "Differential Evolution": {**{p: best_de[p] for p in params}, "RMSE (vol pts)": rmse(best_de["loss"]),
                               "Evaluations": best_de["evaluations"]},
}).T
comparison.astype({"Evaluations": int}).round(4)

### 5.3 Tikhonov regularisation

Along the flat valley, parameters can move a lot for a tiny change of the loss, so a daily recalibration would give
unstable parameters and Greeks. The Tikhonov penalty pulls the solution toward a reference parameter set
(in practice, the previous day's calibration) :

$$
\mathcal{L}_\lambda(\Theta) = \mathcal{L}_{\mathrm{IV}}(\Theta) + \lambda \sum_i \left(\frac{\Theta_i - \Theta_i^{\mathrm{ref}}}{b_i - a_i}\right)^2
$$

where $[a_i, b_i]$ are the parameter bounds. With a single date available, the reference is here a point of the valley
obtained with $\kappa$ fixed to 4, which plays the role of "yesterday's" parameters.

In [ ]:
reference = Calibrator(HestonModel(zc_rate, forward_curve), calibration_loss, fixed={"kappa": 4.0}).run(
    'local', x0={"v0": 0.02, "theta": 0.04, "xi": 1.0, "rho": -0.6})
params_ref = {p: reference[p] for p in params}

rows = [{"lambda": "reference (kappa = 4)", **params_ref, "RMSE (vol pts)": rmse(reference["loss"])}]
for lam in [0.0, 1e-5, 1e-4, 1e-3, 1e-2]:
    tikhonov_calibrator = Calibrator(HestonModel(zc_rate, forward_curve), calibration_loss,
                                     tikhonov=dict(params_ref=params_ref, lam=lam))
    run = tikhonov_calibrator.run('lhs', n_samples=100, n_starts=3, seed=0)
    rows.append({"lambda": lam, **{p: run[p] for p in params}, "RMSE (vol pts)": rmse(run["loss"])})
tikhonov_sweep = pd.DataFrame(rows).set_index("lambda")
tikhonov_sweep.round(4)

$\lambda$ sets the trade-off between fit and stability. The loss is of order $10^{-5}$ (vol units squared), so a useful $\lambda$
makes the penalty worth about 10% of the loss : the parameters move toward the reference at a negligible cost in RMSE.
A larger $\lambda$ lets the penalty dominate and the fit degrades toward the reference's own RMSE.

### 5.4 Retained parameters

The rest of the notebook uses the global minimum found by the Latin Hypercube multi-start.

In [ ]:
for p in params:
    setattr(heston, p, best_lhs[p])
feller_ratio = 2 * heston.kappa * heston.theta / heston.xi**2
print(f"RMSE (vol pts) : {rmse(best_lhs['loss']):.3f} | Feller ratio 2*kappa*theta/xi^2 = {feller_ratio:.3f}")
pd.Series(heston.params, name="Retained Heston parameters").to_frame()

## 6. Heston implied volatility surface

The calibrated model prices every quoted (OTM) option. These prices are then inverted with the same
`BlackScholes` / `ImpliedVolatilitySurface` machinery as the market prices, giving the surface reconstructed by Heston.

In [ ]:
heston_otm_prices = dict()
for exp in market_iv_surface.expiries:
    strikes = np.array(market_iv_surface.strikes[exp])
    heston_otm_prices[exp] = list(heston.price_vanillas(strikes, exp, isCall=strikes > forward_curve.calc_forward(exp)))

heston_iv_surface = ImpliedVolatilitySurface(forward_curve, bs_model)
heston_iv_surface.set_implied_vol_surface(market_iv_surface.strikes, heston_otm_prices)
heston_iv_surface.plot_implied_vol_surface(title="Heston implied volatility surface", show_table=False);

### 6.1 Implied volatility solver : starting point of Newton

`ImpliedVolatilitySurface.get_implied_vol` inverts the Black formula by Newton-Raphson (Brent's method as a fallback).
The number of Newton iterations depends on the starting point $\sigma^{(0)}$. Four starting points are compared:

- **Constant 0.5**: the former default.
- **Corrado-Miller** (1996), the current default (`initial_vol_guess`): an analytical approximation from the price
  alone, exact at the money to first order,
$$
\sigma^{(0)}\sqrt{T} = \frac{\sqrt{2\pi}}{F + K}\left[c - \frac{F - K}{2} + \sqrt{\Big(c - \frac{F - K}{2}\Big)^2 - \frac{(F - K)^2}{\pi}}\;\right],
$$
  with $c$ the undiscounted call price (puts converted by parity) and the square root floored at 0.
- **Nearest neighbour already computed**: the quotes are inverted by increasing expiry, then increasing strike, and each
  inversion starts from the implied volatility of the closest quote already inverted, for the Euclidean norm in the
  plane $(T, k)$ with $k = \ln(K / F(T))$ (Corrado-Miller for the first quote).
- **Market volatility of the same quote**: only when inverting model prices, whose volatilities are close to the market ones.

Two problems: the 142 market prices (giving the market surface of section 3) and the 142 Heston prices of section 6.
A Newton iteration costs one Black price and one vega; the vega calls are counted.

In [ ]:
import time
import ImpliedVolatilitySurface as ivs_module

def invert_all(iv_surface, quotes, starting_point):
    """
    Inverts every quote (in the given order) from the starting point rule ; returns the vols and the solver statistics.
    starting_point(quote, computed) -> float, or None for Corrado-Miller ; computed = [(T, k, vol), ...] already inverted.
    """
    counts = {"newton": 0, "brent": 0}
    original_vega, original_brentq = iv_surface.bs_model.bs_vega, ivs_module.optimize.brentq
    def counted_vega(*args, **kwargs):
        counts["newton"] += 1
        return original_vega(*args, **kwargs)
    def counted_brentq(*args, **kwargs):
        counts["brent"] += 1
        return original_brentq(*args, **kwargs)
    iv_surface.bs_model.bs_vega, ivs_module.optimize.brentq = counted_vega, counted_brentq
    computed, vols, guesses = [], [], []
    start = time.perf_counter()
    try:
        for q in quotes:
            guess = starting_point(q, computed)
            guesses.append(iv_surface.initial_vol_guess(q["T"], q["K"], q["price"], q["isCall"]) if guess is None else guess)
            vol = iv_surface.get_implied_vol(q["T"], q["K"], q["price"], q["isCall"], init_guess=guess, verbose=False)
            vols.append(np.nan if vol is None else vol)
            if vol is not None:
                computed.append((q["T"], q["k"], vol))
    finally:
        del iv_surface.bs_model.bs_vega                     # back to the class method
        ivs_module.optimize.brentq = original_brentq
    return np.array(vols), np.array(guesses), counts, time.perf_counter() - start

def nearest_computed(q, computed):
    """ Implied vol of the closest quote already inverted, Euclidean norm in (T, k) ; None (Corrado-Miller) at first. """
    if not computed:
        return None
    points = np.array(computed)
    distances = np.hypot(points[:, 0] - q["T"], points[:, 1] - q["k"])
    q["neighbour_same_expiry"] = bool(points[np.argmin(distances), 0] == q["T"])
    return float(points[np.argmin(distances), 2])

# The two problems : market prices -> market vols, Heston prices -> Heston vols (OTM quotes, expiry then strike order)
problems = {}
for name, prices, reference in [("Market prices", otm_options, market_iv_surface.implied_vols),
                                ("Heston prices", heston_otm_prices, heston_iv_surface.implied_vols)]:
    quotes = []
    for exp in market_iv_surface.expiries:
        F = forward_curve.calc_forward(exp)
        price_of = dict(zip(market_strikes[exp], prices[exp])) if name == "Market prices" else dict(zip(market_iv_surface.strikes[exp], prices[exp]))
        for K, ref_vol, market_vol in zip(market_iv_surface.strikes[exp], reference[exp], market_iv_surface.implied_vols[exp]):
            quotes.append({"T": exp, "K": K, "k": np.log(K / F), "price": float(price_of[K]), "isCall": bool(K > F),
                           "reference": ref_vol, "market_vol": market_vol})
    problems[name] = quotes

rules = {"Constant 0.5 (former default)": lambda q, c: 0.5,
         "Corrado-Miller (default)": lambda q, c: None,
         "Nearest neighbour already computed (L2 in (T, k))": nearest_computed,
         "Market vol of the same quote": lambda q, c: q["market_vol"]}

rows = []
for problem, quotes in problems.items():
    reference = np.array([q["reference"] for q in quotes])
    for rule_name, rule in rules.items():
        if rule_name.startswith("Market vol") and problem == "Market prices":
            continue                                       # the market vol is the unknown of this problem
        vols, guesses, counts, elapsed = invert_all(market_iv_surface, quotes, rule)
        guess_error = 100 * np.abs(guesses - vols)
        row = {"Problem": problem, "Starting point": rule_name,
               "Newton iterations": counts["newton"], "Iterations per quote": counts["newton"] / len(quotes),
               "Brent fallbacks": counts["brent"], "Failures": int(np.sum(np.isnan(vols))),
               "Time (ms)": 1e3 * elapsed,
               "Median |start - solution| (vol pts)": np.median(guess_error),
               "Max |start - solution| (vol pts)": np.max(guess_error),
               "Max |vol - reference| (vol pts)": 100 * np.nanmax(np.abs(vols - reference))}
        if rule is nearest_computed:
            row["Neighbour in the same expiry (%)"] = 100 * np.mean([q.get("neighbour_same_expiry", True) for q in quotes[1:]])
        rows.append(row)

solver_benchmark = pd.DataFrame(rows).set_index(["Problem", "Starting point"])
solver_benchmark.round(4)

**Reading the table.**
- All the starting points converge to the same volatilities (zero difference at the displayed precision), without any
  Brent fallback nor failure.
- **Iterations.** Corrado-Miller needs the fewest Newton iterations on both problems (2.30 and 2.29 per quote, against
  3.72 and 3.74 from the constant 0.5): its median distance to the solution is 0.04 vol point, against 0.36 for the
  nearest quote already computed and 0.22 for the market volatility of the same quote (the Heston calibration error).
  Newton needs at least two vega evaluations per quote (one step, then the convergence check), so 2.3 is close to the
  minimum.
- **Nearest neighbour.** With the Euclidean norm in $(T, k)$, the nearest quote already computed lies in the same expiry
  for 91% of the quotes: the strike spacing in log-moneyness is smaller than the gap between expiries. Its volatility
  is 0.36 vol point away from the solution in the median, because the smile is steep.
- **Time.** Corrado-Miller is not the fastest in time (259 and 254 ms, against 247 and 242 ms for the nearest
  neighbour and 213 ms for the market volatility): its starting point needs the forward $F(T)$ and the discount factor,
  i.e. about the cost of one more Black price. The three warm starts gain 20 to 35% of time on the former default.
- **Largest gaps.** The largest distances to the solution (3.3 vol points for Corrado-Miller, 3.1 for the nearest
  neighbour) are on deep out-of-the-money short-dated quotes, where Newton still converges.

## 7. Market vs Heston

### 7.1 Surfaces

In [ ]:
K_range = (0.8 * max(min(k) for k in market_iv_surface.strikes.values()),
           1.2 * min(max(k) for k in market_iv_surface.strikes.values()))
K, T, market_vols = market_iv_surface.vol_grid(K_range=K_range)
_, _, heston_vols = heston_iv_surface.vol_grid(K_range=K_range)
z_min, z_max = 100 * min(market_vols.min(), heston_vols.min()), 100 * max(market_vols.max(), heston_vols.max())

fig = plt.figure(figsize=(20, 6.5))
for i, (vols, title) in enumerate([(market_vols, "Market"), (heston_vols, "Heston")]):
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    ax.plot_surface(K, T, 100 * vols, cmap=cm.coolwarm, vmin=z_min, vmax=z_max, linewidth=0, antialiased=True)
    ax.set_zlim(z_min, z_max)
    ax.set_title(f"{title} implied volatility")
    ax.set_xlabel("Strike (K)")
    ax.set_ylabel("Expiry (T)")
    ax.set_zlabel("Vol (%)", labelpad=8)

ax = fig.add_subplot(1, 3, 3)
diff = 100 * (heston_vols - market_vols)
bound = np.nanmax(np.abs(diff))
im = ax.pcolormesh(K, T, diff, cmap="RdBu_r", vmin=-bound, vmax=bound, shading="auto")
fig.colorbar(im, ax=ax, label="Heston - Market (vol pts)")
ax.set_title("Implied volatility error")
ax.set_xlabel("Strike (K)")
ax.set_ylabel("Expiry (T)")
plt.tight_layout()
plt.show()

### 7.2 Smiles per expiry

In [ ]:
expiries = market_iv_surface.expiries
n_cols = 4
n_rows = int(np.ceil(len(expiries) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.2 * n_rows))
for ax, exp in zip(axes.flat, expiries):
    strikes = market_iv_surface.strikes[exp]
    K_fine = np.linspace(strikes[0], strikes[-1], 60)
    heston_fine = heston.price_vanillas(K_fine, exp, isCall=K_fine > forward_curve.calc_forward(exp))
    heston_smile = [heston_iv_surface.get_implied_vol(exp, k, p, bool(k > forward_curve.calc_forward(exp)), verbose=False)
                    for k, p in zip(K_fine, heston_fine)]
    ax.scatter(strikes, 100 * np.array(market_iv_surface.implied_vols[exp]), color="black", s=15, label="Market", zorder=5)
    ax.plot(K_fine, [100 * v if v is not None else np.nan for v in heston_smile], color="red", label="Heston")
    ax.axvline(forward_curve.calc_forward(exp), color="grey", linestyle=":", linewidth=1)
    ax.set_title(f"T = {exp:.2f}y")
    ax.grid(True, linestyle='--', alpha=0.5)
for ax in axes.flat[len(expiries):]:
    ax.axis("off")
axes.flat[0].legend()
fig.supxlabel("Strike")
fig.supylabel("Implied volatility (%)")
plt.tight_layout()
plt.show()

### 7.3 Calibration errors

In [ ]:
rows = []
for exp in market_iv_surface.expiries:
    for K_, mkt, hst in zip(market_iv_surface.strikes[exp], market_iv_surface.implied_vols[exp], heston_iv_surface.implied_vols[exp]):
        rows.append({"Expiry": exp, "Strike": K_, "Market IV (%)": 100 * mkt, "Heston IV (%)": 100 * hst})
errors = pd.DataFrame(rows)
errors["Error (vol pts)"] = errors["Heston IV (%)"] - errors["Market IV (%)"]

summary = errors.groupby("Expiry")["Error (vol pts)"].agg(
    RMSE=lambda e: np.sqrt(np.mean(e**2)), MAE=lambda e: np.mean(np.abs(e)), Max=lambda e: e.abs().max())
print(f"Global implied vol RMSE : {np.sqrt(np.mean(errors['Error (vol pts)']**2)):.3f} vol pts")
summary.round(3)

## 8. Static no-arbitrage conditions

A surface of call prices $(K, T) \mapsto C(K, T)$ is free of static arbitrage if and only if
- **monotonicity**: $-D(T) \le \partial_K C(K, T) \le 0$;
- **butterfly**: $\partial_{KK} C(K, T) \ge 0$ (the risk-neutral density $\partial_{KK} C / D(T)$ is non-negative);
- **calendar**: $T \mapsto C(F(T)e^k, T) / (D(T)F(T))$ is non-decreasing at fixed forward log-moneyness $k$.

With the normalised price $\hat c(\kappa, T) = C(\kappa F(T), T) / (D(T) F(T))$ and the moneyness $\kappa = K/F(T)$,
$\partial_K C = D(T)\,\partial_\kappa \hat c$: the three conditions read $-1 \le \partial_\kappa \hat c \le 0$, $\hat c$ convex
in $\kappa$, and $\hat c$ non-decreasing in $T$ at fixed $\kappa$. On a discrete grid they are exactly conditions on the
slopes between neighbouring points: slopes in $[-1, 0]$, non-decreasing slopes, non-decreasing prices in $T$.

Six surfaces are checked on a grid of 71 log-moneynesses $k \in [-40\%, 30\%]$ and 61 maturities (the 13 listed
expiries and 48 dates in between):
1. market surface, **linear in strike + V2T** (`get_vol`, method `v2t`: the interpolation used for the plots);
2. market surface, **SVI** (one smile per expiry, total variance linear in $T$ at fixed $k$);
3. market surface, **SSVI** (arbitrage-free parametrisation);
4. **Heston prices** (Lewis formula), arbitrage-free by construction up to the quadrature error;
   computed with the default quadrature (4,000 nodes up to $u = 1000$) and with a fine one (60,000 nodes up to
   $u = 3000$);
5. **Heston implied surface** interpolated linearly in strike + V2T, as the market one.

For the volatility surfaces, $\hat c = N(d_1) - \kappa N(d_2)$ with $d_{1,2} = -\ln\kappa / \sqrt{w} \pm \sqrt{w}/2$ and
$w = \sigma^2 T$. A violation is counted when a condition fails by more than $10^{-7}$ (0.0008 index point on a
forward of 8000).

In [ ]:
k_grid = np.round(np.arange(-0.40, 0.3001, 0.01), 4)
T_check = np.unique(np.concatenate([market_iv_surface.expiries, np.linspace(0.05, 4.85, 48)]))
tolerance = 1e-7

def normalised_call(k, w):
    """ Black call price divided by D(T) F(T), from log-moneyness k = ln(K/F) and total variance w. """
    s = np.sqrt(np.maximum(w, 1e-16))
    d1 = -k / s + s / 2
    return norm.cdf(d1) - np.exp(k) * norm.cdf(d1 - s)

def vol_surface_prices(iv_surface, method):
    """ Normalised call prices on (T_check x k_grid) from an implied volatility interpolation. """
    out = np.empty((len(T_check), len(k_grid)))
    for i, T in enumerate(T_check):
        F = forward_curve.calc_forward(T)
        vols = np.array([float(iv_surface.get_vol(T, F * np.exp(k), method=method)) for k in k_grid])
        out[i] = normalised_call(k_grid, vols**2 * T)
    return out

def heston_prices(model, **quadrature):
    """ Normalised call prices on (T_check x k_grid) from the Heston model (Lewis formula, optional quadrature settings). """
    out = np.empty((len(T_check), len(k_grid)))
    for i, T in enumerate(T_check):
        F, D = forward_curve.calc_forward(T), np.exp(-zc_rate(T) * T)
        out[i] = model.price_vanillas(F * np.exp(k_grid), T, isCall=True, **quadrature) / (D * F)
    return out

def arbitrage_report(c):
    """ Violations of the three conditions on a (T x k) grid of normalised call prices. """
    kappa = np.exp(k_grid)
    slopes = np.diff(c, axis=1) / np.diff(kappa)                      # d c / d kappa between neighbours
    monotonicity = np.maximum(slopes - 0.0, -1.0 - slopes)            # > 0 : slope outside [-1, 0]
    butterfly = -np.diff(slopes, axis=1)                              # > 0 : decreasing slopes (concavity)
    calendar = -np.diff(c, axis=0)                                    # > 0 : price decreasing in T at fixed k
    return {name: (values > tolerance, np.maximum(values.max(), 0.0))
            for name, values in [("monotonicity", monotonicity), ("butterfly", butterfly), ("calendar", calendar)]}

start = time.time()
surfaces = {"Market, linear + V2T": vol_surface_prices(market_iv_surface, "v2t"),
            "Market, SVI": vol_surface_prices(market_iv_surface, "svi"),
            "Market, SSVI": vol_surface_prices(market_iv_surface, "ssvi"),
            "Heston prices (Lewis)": heston_prices(heston),
            "Heston prices (Lewis, fine quadrature)": heston_prices(heston, u_max=3000.0, n_u=60_000),
            "Heston implied, linear + V2T": vol_surface_prices(heston_iv_surface, "v2t")}
reports = {name: arbitrage_report(c) for name, c in surfaces.items()}
print(f"{time.time() - start:.1f}s")

rows = {}
for name, report in reports.items():
    row = {}
    for condition, (violated, worst) in report.items():
        row[f"{condition} : violations"] = f"{violated.sum()} / {violated.size}"
        row[f"{condition} : worst (x 1e-4)"] = 1e4 * worst
    rows[name] = row
arbitrage_table = pd.DataFrame(rows).T
arbitrage_table

**Where are the violations?** Each panel shows the grid points where a condition fails (monotonicity and butterfly at
the middle of the strike interval, calendar between two maturities).

In [ ]:
fig, axes = plt.subplots(1, len(reports), figsize=(4.2 * len(reports), 4), sharey=True)
markers = {"monotonicity": ("tab:blue", "s"), "butterfly": ("tab:red", "o"), "calendar": ("tab:green", "^")}
for ax, (name, report) in zip(axes, reports.items()):
    for condition, (violated, _) in report.items():
        if condition == "calendar":
            T_points, k_points = (T_check[:-1] + T_check[1:]) / 2, k_grid
        elif condition == "monotonicity":
            T_points, k_points = T_check, (k_grid[:-1] + k_grid[1:]) / 2
        else:
            T_points, k_points = T_check, k_grid[1:-1]
        ii, jj = np.nonzero(violated)
        color, marker = markers[condition]
        ax.scatter(100 * k_points[jj], T_points[ii], s=12, color=color, marker=marker,
                   label=f"{condition} ({len(ii)})")
    for T in market_iv_surface.expiries:
        ax.axhline(T, color="grey", linewidth=0.3)
    ax.set_xlim(100 * k_grid[0], 100 * k_grid[-1]); ax.set_ylim(0, 5)
    ax.set_title(name, fontsize=10); ax.set_xlabel("log-moneyness k (%)")
    ax.legend(fontsize=7, loc="upper left")
axes[0].set_ylabel("Maturity T (years)")
plt.tight_layout(); plt.show()

**Risk-neutral densities.** The butterfly condition is the positivity of the density of $S_T$,
$\partial_{KK} C / D(T)$ (Breeden-Litzenberger), shown here in moneyness ($\partial_{\kappa\kappa}\hat c$) at three maturities.
A linear interpolation of the volatility in strike has kinks at the quoted strikes: the density jumps there.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
kappa = np.exp(k_grid)
for ax, T_target in zip(axes, [0.1014, 1.0986, 3.8411]):
    i = int(np.argmin(np.abs(T_check - T_target)))
    for name, c in surfaces.items():
        slopes = np.diff(c[i]) / np.diff(kappa)
        density = np.diff(slopes) / ((kappa[2:] - kappa[:-2]) / 2)
        ax.plot(100 * k_grid[1:-1], density, label=name, linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(f"T = {T_check[i]:.2f}y"); ax.set_xlabel("log-moneyness k (%)")
    ax.grid(True, linestyle="--", alpha=0.5)
axes[0].set_ylabel("density of S_T / F in moneyness"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading the results.**
- **SSVI** satisfies the three conditions everywhere on the grid, as expected from its arbitrage-free parametrisation.
- **Linear + V2T** (market: 139 butterfly violations, Heston implied: 107) never violates monotonicity nor the calendar
  condition, but violates the butterfly condition along vertical lines at $k \approx -35\%, -27\%, -13\%, -3\%$. Each
  line is the lowest quoted strike of a group of expiries (7800 from 3 to 10 months, 7000 from 1 to 1.8 years, 6000
  and 5600 beyond), where `get_vol` switches from a linear interpolation with a negative slope to a flat extrapolation
  of the volatility: the kink makes the density negative (down to $-15$ at 3.8 years). The same violations on the
  Heston implied surface, which comes from an arbitrage-free model, show that they are created by the interpolation,
  not by the data. The kinks of the linear interpolation at the quoted strikes are also visible as spikes of the density.
- **SVI** has no butterfly violation but 12 calendar violations and one monotonicity violation, all around
  $T \in [1.6, 1.85]$ years and $k \ge 27\%$: the SVI slices of 1.60 and 1.85 years cross in the right wing (the 1.60-year
  slice has an unusual positive skew parameter $\rho = 0.78$). Independent slices do not guarantee the calendar condition.
- **Heston prices** are arbitrage-free by construction: the 45 monotonicity violations of the default Lewis quadrature
  are all at $T \le 0.1$ and $k \le -14\%$, where the prices are almost zero and the quadrature error (up to $3.5 \times 10^{-6}$
  of the forward) even gives slightly negative prices; with the fine quadrature there is no violation left.

The surface used for the plots (linear + V2T) is therefore fine for display but not for pricing with a local volatility
model, which needs the second derivative in strike: SSVI is used for this purpose in `lsv_test.ipynb`.

## 9. Monte Carlo sanity check

The semi-analytical Heston price is compared with a Monte Carlo price obtained from `HestonModel.generate_paths`
(Euler scheme, full truncation) for an ATM call at the 1-year expiry.

In [ ]:
T_mc = listed_expiries[6]
K_mc = 8000.0
S_paths, v_paths = heston.generate_paths(T_mc, M=250, N=100_000, seed=42)
payoff = np.maximum(S_paths[-1] - K_mc, 0.0)
df_mc = np.exp(-zc_rate(T_mc) * T_mc)
mc_price = df_mc * payoff.mean()
mc_std = df_mc * payoff.std() / np.sqrt(len(payoff))

print(f"T = {T_mc:.4f}y, K = {K_mc}")
print(f"Semi-analytical price : {heston.price_vanilla(K_mc, T_mc):.2f}")
print(f"Monte Carlo price     : {mc_price:.2f} +/- {1.96 * mc_std:.2f} (95%)")
print(f"E[S_T] (MC) = {S_paths[-1].mean():.2f} vs forward F(0,T) = {forward_curve.calc_forward(T_mc):.2f}")

## 10. Time-dependent Heston

With five constant parameters, the Heston model cannot fit the whole surface: the ATM skew of the model is almost
constant at short maturities, whereas the market one explodes, and a single pair $(\kappa, \theta)$ must describe the
whole term structure of variance (section 7.3: errors of 0.5 vol pt at 9 days and 0.6 - 0.8 vol pt beyond 2.8 years).
Since the data cannot change, the model is enriched (Mikhailov & Nögel, 2003):
$$
dv_t = \kappa\,(\theta(t) - v_t)\,dt + \xi(t)\sqrt{v_t}\,dZ_t, \qquad d\langle W, Z\rangle_t = \rho(t)\,dt,
$$
with $\theta, \xi, \rho$ constant on each interval $(T_{j-1}, T_j]$ between two listed expiries, and $v_0$, $\kappa$ global.

**Pricing.** The characteristic function of $X_T = \ln(S_T/F(0,T))$ is still $\exp(C(u,T) + D(u,T)\,v_0)$: the Riccati
equations are solved backward from $T$ to 0, interval by interval. On an interval of length $\tau$ with terminal values
$(C_0, D_0)$, $\beta = \kappa - i\rho\xi u$, $d = \sqrt{\beta^2 + \xi^2(u^2 + iu)}$ and $g = (\beta - d - \xi^2 D_0)/(\beta + d - \xi^2 D_0)$:
$$
D = D_0 + \frac{\beta - d - \xi^2 D_0}{\xi^2}\,\frac{1 - e^{-d\tau}}{1 - g e^{-d\tau}}, \qquad
C = C_0 + \frac{\kappa\theta}{\xi^2}\Big[(\beta - d)\tau - 2\ln\frac{1 - g e^{-d\tau}}{1 - g}\Big].
$$
The Lewis formula is unchanged; a price at $T_j$ costs $j$ evaluations of these formulas instead of one.

**Calibration** (`Model/Heston.py`, class `TimeDependentHestonModel`, $\kappa$ and $v_0$ started from the constant calibration of section 5.4):
1. **bootstrap**: for $j = 1, \dots, 13$, the parameters of interval $j$ are fitted to the smile of $T_j$ alone, the earlier
   intervals being fixed ($v_0$ is fitted with the first expiry, $\kappa$ is kept). Fast, but greedy: each interval absorbs
   the errors left by the previous ones;
2. **global**: all the parameters ($v_0$, $\kappa$ and the time-dependent ones) jointly on all the expiries (same loss as
   section 5), started from the bootstrap;
3. **global with smoothness**: penalty $\lambda \sum_j \sum_p \big((p_j - p_{j-1})/\text{range}_p\big)^2$ with $\lambda = 10^{-4}$,
   to avoid parameters that jump from one interval to the next.

Two families are compared: $\theta(t)$ only (15 parameters) and $(\theta, \xi, \rho)(t)$ (41 parameters).

In [ ]:
import copy
from Heston import TimeDependentHestonModel

expiries = market_iv_surface.expiries
td_check = TimeDependentHestonModel.from_heston(heston, expiries)
u_check = np.linspace(1e-8, 1000, 4000) - 0.5j
cf_gap = max(np.max(np.abs(td_check.characteristic_function(u_check, T) - heston.characteristic_function(u_check, T)))
             for T in [0.02, 0.6, 3.0, 6.0])
print(f"Same parameters on every interval : max |phi_TD - phi_Heston| = {cf_gap:.1e}")

### 10.1 Bootstrap

In [ ]:
td_theta = TimeDependentHestonModel.from_heston(heston, expiries)
boot_theta = td_theta.calibrate_bootstrap(calibration_loss, free=("theta",))
td_full = TimeDependentHestonModel.from_heston(heston, expiries)
boot_full = td_full.calibrate_bootstrap(calibration_loss)
boot_full.round(4)

### 10.2 Global calibration
The global runs use finite-difference gradients over 15 or 41 parameters: several minutes each. The run with smoothness
starts from the global optimum without penalty.

In [ ]:
td_theta_global = copy.copy(td_theta)
td_theta_global.calibrate_global(calibration_loss, free=("theta",))
td_full_global = copy.copy(td_full)
td_full_global.calibrate_global(calibration_loss)
td_full_smooth = copy.copy(td_full_global)
td_full_smooth.calibrate_global(calibration_loss, smoothness=1e-4, maxiter=150);

### 10.3 Comparison
Implied volatility RMSE (after inversion of the model prices, vol points) per expiry and on the whole surface. The number
of free parameters is given in brackets.

In [ ]:
td_models = {"Heston (5)": heston,
             "theta(t), bootstrap (14)": td_theta, "theta(t), global (15)": td_theta_global,
             "(theta, xi, rho)(t), bootstrap (40)": td_full, "(theta, xi, rho)(t), global (41)": td_full_global,
             "(theta, xi, rho)(t), global + smoothness (41)": td_full_smooth}
td_reports = {name: calibration_loss.report(model) for name, model in td_models.items()}
td_comparison = pd.DataFrame({name: by_expiry["IV RMSE (vol pts)"] for name, (_, by_expiry) in td_reports.items()})
td_comparison.index = [f"{T:.3f}" for T in td_comparison.index]
td_comparison.loc["Global RMSE"] = [summary["IV RMSE (vol pts)"] for summary, _ in td_reports.values()]
td_comparison.loc["Global max"] = [summary["IV max (vol pts)"] for summary, _ in td_reports.values()]
td_comparison.round(3)

**Term structure of the parameters** (step functions on the intervals between listed expiries).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
edges = np.concatenate([[0.0], expiries])
for name, model, style in [("constant Heston", heston, ":"), ("global", td_full_global, "-"),
                           ("global + smoothness", td_full_smooth, "--")]:
    for ax, p in zip(axes, ["theta", "xi", "rho"]):
        values = np.broadcast_to(getattr(model, p), (len(expiries),))
        ax.stairs(values, edges, linestyle=style, linewidth=1.6, label=name)
for ax, p in zip(axes, ["theta", "xi", "rho"]):
    ax.set_title(p); ax.set_xlabel("T (years)"); ax.grid(True, linestyle="--", alpha=0.5)
axes[2].axhline(0, color="black", linewidth=0.6)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"global + smoothness : v0 = {td_full_smooth.v0:.4f}, kappa = {td_full_smooth.kappa:.4f}")
td_full_smooth.segment_table().round(4)

**Smiles.** Market and model implied volatilities at the quoted strikes: constant Heston against the time-dependent model
with smoothness.

In [ ]:
heston_vols = calibration_loss.model_implied_vols(heston)
td_vols = calibration_loss.model_implied_vols(td_full_smooth)
n_cols = 4
n_rows = int(np.ceil(len(expiries) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.2 * n_rows))
for ax, q, hv, tv in zip(axes.flat, calibration_loss.quotes, heston_vols, td_vols):
    ax.plot(q["strikes"], 100 * q["vols"], "o", color="black", markersize=3, label="market")
    ax.plot(q["strikes"], 100 * hv, "-", linewidth=1.3, label="Heston")
    ax.plot(q["strikes"], 100 * tv, "--", linewidth=1.3, label="time-dependent Heston")
    ax.set_title(f"T = {q['expiry']:.3f}y", fontsize=10); ax.grid(True, linestyle="--", alpha=0.5)
for ax in axes.flat[len(expiries):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Implied volatility surface.** The prices of the time-dependent model with smoothness are inverted at the quoted points
and interpolated as in section 7.1, then compared with the market and the constant Heston surfaces (same colour scale
for the two error maps).

In [ ]:
td_otm_prices = dict()
for exp in market_iv_surface.expiries:
    strikes = np.array(market_iv_surface.strikes[exp])
    td_otm_prices[exp] = list(td_full_smooth.price_vanillas(strikes, exp, isCall=strikes > forward_curve.calc_forward(exp)))
td_iv_surface = ImpliedVolatilitySurface(forward_curve, bs_model)
td_iv_surface.set_implied_vol_surface(market_iv_surface.strikes, td_otm_prices)

K_grid, T_grid, market_grid = market_iv_surface.vol_grid(K_range=K_range)
_, _, heston_grid = heston_iv_surface.vol_grid(K_range=K_range)
_, _, td_grid = td_iv_surface.vol_grid(K_range=K_range)
grids = [(market_grid, "Market"), (heston_grid, "Heston (constant)"), (td_grid, "Time-dependent Heston (smoothness)")]
z_min = 100 * min(g.min() for g, _ in grids)
z_max = 100 * max(g.max() for g, _ in grids)

fig = plt.figure(figsize=(20, 12))
for i, (vols, title) in enumerate(grids):
    ax = fig.add_subplot(2, 3, i + 1, projection='3d')
    ax.plot_surface(K_grid, T_grid, 100 * vols, cmap=cm.coolwarm, vmin=z_min, vmax=z_max, linewidth=0, antialiased=True)
    ax.set_zlim(z_min, z_max)
    ax.set_title(f"{title} implied volatility")
    ax.set_xlabel("Strike (K)"); ax.set_ylabel("Expiry (T)"); ax.set_zlabel("Vol (%)", labelpad=8)

errors_grid = [(100 * (heston_grid - market_grid), "Heston (constant) - Market"),
               (100 * (td_grid - market_grid), "Time-dependent Heston - Market")]
bound = max(np.nanmax(np.abs(e)) for e, _ in errors_grid)
for i, (diff, title) in enumerate(errors_grid):
    ax = fig.add_subplot(2, 3, 4 + i)
    im = ax.pcolormesh(K_grid, T_grid, diff, cmap="RdBu_r", vmin=-bound, vmax=bound, shading="auto")
    fig.colorbar(im, ax=ax, label="vol pts")
    ax.set_title(f"{title} (grid RMSE {np.sqrt(np.nanmean(diff**2)):.3f} vol pts)")
    ax.set_xlabel("Strike (K)"); ax.set_ylabel("Expiry (T)")
plt.tight_layout()
plt.show()

**Monte Carlo check.** Prices of the time-dependent model (Lewis formula) against an Euler simulation with the parameters
of the interval of each step (400 steps, 200,000 antithetic paths).

In [ ]:
T_mc = expiries[8]
strikes_mc = np.array([6500.0, 7500.0, 8000.0, 8500.0, 9500.0])
S_paths, _ = td_full_smooth.generate_paths(T_mc, M=400, N=200_000, seed=7, antithetic=True)
df_mc = np.exp(-zc_rate(T_mc) * T_mc)
is_call_mc = strikes_mc >= forward_curve.calc_forward(T_mc)
payoffs = np.where(is_call_mc[:, None], np.maximum(S_paths[-1][None, :] - strikes_mc[:, None], 0.0),
                   np.maximum(strikes_mc[:, None] - S_paths[-1][None, :], 0.0)) * df_mc
pd.DataFrame({"Strike": strikes_mc, "Type": np.where(is_call_mc, "call", "put"),
              "Lewis": td_full_smooth.price_vanillas(strikes_mc, T_mc, is_call_mc),
              "Monte Carlo": payoffs.mean(axis=1),
              "MC std err": payoffs.std(axis=1) / np.sqrt(payoffs.shape[1])}).round(3)

**Reading the results.**
- **The pricing is exact**: with the same parameters on every interval, the characteristic function equals the constant
  Heston one to $3 \times 10^{-16}$.
- **$\theta(t)$ alone** (15 parameters) lowers the RMSE from 0.421 to 0.265 vol pt and the maximum error from 1.75 to
  1.08 vol pts: it fixes the term structure of the ATM level (long maturities: 0.64 - 0.81 down to 0.31 - 0.55), but not
  the short-maturity skew (0.20 vol pt left at 9 days), which depends on $\xi$ and $\rho$.
- **$(\theta, \xi, \rho)(t)$** (41 parameters) lowers the RMSE to 0.163 (bootstrap), 0.118 (global) and 0.148 vol pt
  (global with smoothness), i.e. 2.8 to 3.6 times less than the constant model, with a maximum error of 0.5 vol pt. The
  9-day smile, the worst one of the constant model after the long maturities, is fitted to 0.04 vol pt without penalty.
- **The bootstrap is greedy**: each interval absorbs the errors of the previous ones, and $\xi$ reaches its bound (5) beyond
  1.85 years. The global calibration spreads the error over all the expiries.
- **Over-parametrisation**: without penalty, the parameters jump from one interval to the next; the smoothness penalty
  gives regular term structures ($\rho$ between $-0.70$ and $-0.46$, $\theta$ decreasing then increasing, $\xi$ falling to 0.25
  around 0.7 year, then rising) for 0.03 vol pt of RMSE. The global run with smoothness stops at the iteration limit (150) and
  takes 7 minutes: finite-difference gradients over 41 parameters are the bottleneck.
- **The price of the fit**: $\xi(t)$ reaches 2 - 3.3 beyond 1.6 years, so the Feller ratio falls to 0.02 - 0.05. The
  variance spends more time near zero, which matters for path-dependent products, and the full-truncation Euler scheme
  becomes biased: at 1.6 years, the Monte Carlo call prices exceed the Lewis ones by 2 - 3 standard errors with 400 steps.
  The gap decreases with the number of steps (at-the-money call: +15.1 with 100 steps, +4.6 with 400, +3.1 with 1,600),
  so it comes from the discretisation and not from the model.

The time-dependent model removes the structural error of the constant model on vanillas, but the constant model is not
replaced for free: the dynamics (forward skew, vol of vol) depend on the parameter term structure, which the vanilla
surface does not pin down.

## 11. Overfitting: out-of-sample and stability tests

The time-dependent models of section 10 fit the 142 quotes much better, with up to 41 parameters. Two tests tell a better
model from an overfitted one:
1. **out-of-sample**: calibrate without some quotes and measure the error on them. An overfitted model has a small in-sample
   error and a large out-of-sample one. Two splits:
   - *leave-one-expiry-out*: each of the 13 expiries is left out in turn (13 calibrations); the intervals of the
     time-dependent models then end at the 12 remaining expiries, so the left-out expiry is priced with the parameters
     of the merged interval (interpolation in time; extrapolation for the last expiry);
   - *strike holdout*: one strike out of two is left out at every expiry (2 calibrations, on 77 and 65 quotes);
2. **stability**: the market implied volatilities are perturbed by an i.i.d. Gaussian noise of 0.1 vol pt (20 draws), of
   the order of a bid-ask half-spread, and the models are recalibrated. A model that fits the noise moves its surface and its
   parameters with it.

Every calibration reruns the whole procedure of section 10 on the modified data (bootstrap, global, global with smoothness,
started from the constant model), so that no information from the left-out quotes leaks into the starting point. The only
exception is the constant model, run locally from the global minimum of section 5 (unique minimum found by the Latin
Hypercube and Differential Evolution). The 35 calibrations (5 models each) are independent and run in parallel
(`Model/calibration_validation.py`, joblib): about 50 minutes on 20 cores.

In [ ]:
import calibration_validation as cv

short = {"Heston": "Heston (5)", "theta(t), global": "theta(t) global (15)",
         "(theta, xi, rho)(t), bootstrap": "full bootstrap (40)", "(theta, xi, rho)(t), global": "full global (41)",
         "(theta, xi, rho)(t), global + smoothness": "full smooth (41)"}
in_sample = {"Heston": heston, "theta(t), global": td_theta_global, "(theta, xi, rho)(t), bootstrap": td_full,
             "(theta, xi, rho)(t), global": td_full_global, "(theta, xi, rho)(t), global + smoothness": td_full_smooth}
n_exp = len(calibration_loss.quotes)
vol_noise, n_draws = 0.001, 20

jobs = [(("loo", j), cv.subset_loss(calibration_loss, [i for i in range(n_exp) if i != j]), heston) for j in range(n_exp)]
jobs += [(("strikes", parity), cv.strike_split_loss(calibration_loss, parity), heston) for parity in (0, 1)]
jobs += [(("noise", s), cv.perturbed_loss(calibration_loss, bs_model, vol_noise, seed=s), heston) for s in range(n_draws)]
start = time.time()
validation = cv.run_parallel(jobs)
print(f"{len(jobs)} calibrations of the 5 models in {time.time() - start:.0f}s")

in_sample_vols = {name: calibration_loss.model_implied_vols(model) for name, model in in_sample.items()}
in_sample_errors = {name: [v - q["vols"] for v, q in zip(vols, calibration_loss.quotes)]
                    for name, vols in in_sample_vols.items()}

### 11.1 Leave-one-expiry-out
IV RMSE (vol pts) at each expiry: in-sample (calibration on the 13 expiries) and out-of-sample (calibration on the 12 others).

In [ ]:
loo_out = {name: [] for name in cv.FAMILY}
for j in range(n_exp):
    left_out = cv.subset_loss(calibration_loss, [j])
    for name in cv.FAMILY:
        model = cv.build_model(validation[("loo", j)][name], heston)
        loo_out[name].append(cv.iv_errors(model, left_out)[0])

loo_table = {}
for name in cv.FAMILY:
    loo_table[(short[name], "in")] = [cv.rmse([e]) for e in in_sample_errors[name]] + [cv.rmse(in_sample_errors[name])]
    loo_table[(short[name], "out")] = [cv.rmse([e]) for e in loo_out[name]] + [cv.rmse(loo_out[name])]
loo_table = pd.DataFrame(loo_table, index=[f"{q['expiry']:.3f}" for q in calibration_loss.quotes] + ["All"])
loo_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
T_axis = [q["expiry"] for q in calibration_loss.quotes]
for name in cv.FAMILY:
    line, = ax.plot(T_axis, loo_table[(short[name], "out")].iloc[:-1], "o-", linewidth=1.3, label=f"{short[name]} out")
    ax.plot(T_axis, loo_table[(short[name], "in")].iloc[:-1], ":", color=line.get_color(), linewidth=1.0)
ax.set_xscale("log"); ax.set_xlabel("left-out expiry T (years, log scale)"); ax.set_ylabel("IV RMSE (vol pts)")
ax.set_title("Leave-one-expiry-out: out-of-sample (solid) vs in-sample (dotted) error")
ax.grid(True, linestyle="--", alpha=0.5); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 11.2 Strike holdout
One strike out of two is used for the calibration, the others for the test; the two complementary splits are pooled.

In [ ]:
strike_rows = {}
for name in cv.FAMILY:
    train, test = [], []
    for parity in (0, 1):
        split = cv.strike_split_loss(calibration_loss, parity)
        errors = cv.iv_errors(cv.build_model(validation[("strikes", parity)][name], heston), calibration_loss)
        for e, q in zip(errors, split.quotes):
            train.append(e[q["weights"] == 1]); test.append(e[q["weights"] == 0])
    strike_rows[short[name]] = {"in-sample, all quotes": cv.rmse(in_sample_errors[name]),
                                "train quotes": cv.rmse(train), "test quotes": cv.rmse(test),
                                "test / train": cv.rmse(test) / cv.rmse(train)}
strike_table = pd.DataFrame(strike_rows).T
strike_table.round(3)

### 11.3 Stability under noise
For each draw: error against the noisy quotes (the calibration target), error against the original quotes, and
displacement of the fitted surface (RMSE between the implied volatilities of the model calibrated on noisy data and of the
model calibrated on the original data). For a linear least-squares fit with $k$ parameters and $n$ quotes, the noise moves the
fitted values by $\sigma\sqrt{k/n}$ on average and the residual against the noisy data shrinks to $\sqrt{e^2 + \sigma^2(1 - k/n)}$
where $e$ is the original misfit: these are given as references.

In [ ]:
n_quotes = calibration_loss.n_quotes
noise_vols = {name: [] for name in cv.FAMILY}
noise_rows = {}
for name in cv.FAMILY:
    to_noisy, to_clean, displacement = [], [], []
    for s in range(n_draws):
        noisy = cv.perturbed_loss(calibration_loss, bs_model, vol_noise, seed=s)
        vols = calibration_loss.model_implied_vols(cv.build_model(validation[("noise", s)][name], heston))
        noise_vols[name].append(vols)
        to_noisy.append(cv.rmse([v - q["vols"] for v, q in zip(vols, noisy.quotes)]))
        to_clean.append(cv.rmse([v - q["vols"] for v, q in zip(vols, calibration_loss.quotes)]))
        displacement.append(cv.rmse([v - w for v, w in zip(vols, in_sample_vols[name])]))
    k = cv.FAMILY[name]
    e = cv.rmse(in_sample_errors[name])
    noise_rows[short[name]] = {"in-sample (original)": e,
                               "fit to noisy quotes": np.sqrt(np.mean(np.square(to_noisy))),
                               "reference sqrt(e^2 + s^2 (1 - k/n))": np.sqrt(e**2 + (100 * vol_noise)**2 * (1 - k / n_quotes)),
                               "error vs original quotes": np.sqrt(np.mean(np.square(to_clean))),
                               "surface displacement": np.sqrt(np.mean(np.square(displacement))),
                               "reference s sqrt(k/n)": 100 * vol_noise * np.sqrt(k / n_quotes)}
noise_table = pd.DataFrame(noise_rows).T
print(f"Noise : {100 * vol_noise:.1f} vol pt, {n_draws} draws, n = {n_quotes} quotes (all values in vol pts)")
noise_table.round(3)

**Parameters under noise.** Standard deviation of each parameter over the 20 draws (for the time-dependent parameters:
median and maximum over the 13 intervals), then the 20 recalibrated term structures (thin lines) around the original one
(thick line).

In [ ]:
def draws(name, p):
    return np.array([np.broadcast_to(validation[("noise", s)][name][p], (n_exp,)) for s in range(n_draws)])

param_rows = {}
for name in cv.FAMILY:
    row = {"v0": np.std([validation[("noise", s)][name]["v0"] for s in range(n_draws)]),
           "kappa": np.std([validation[("noise", s)][name]["kappa"] for s in range(n_draws)])}
    for p in ["theta", "xi", "rho"]:
        std = draws(name, p).std(axis=0)
        row[f"{p} median"], row[f"{p} max"] = np.median(std), std.max()
    param_rows[short[name]] = row
param_dispersion = pd.DataFrame(param_rows).T
display(param_dispersion.round(4))

fig, axes = plt.subplots(2, 3, figsize=(17, 7.5), sharex=True)
edges = np.concatenate([[0.0], expiries])
for row, name in zip(axes, ["(theta, xi, rho)(t), global", "(theta, xi, rho)(t), global + smoothness"]):
    for ax, p in zip(row, ["theta", "xi", "rho"]):
        for values in draws(name, p):
            ax.stairs(values, edges, color="tab:blue", alpha=0.25, linewidth=0.8)
        ax.stairs(getattr(in_sample[name], p), edges, color="black", linewidth=2.0)
        ax.set_title(f"{short[name]} : {p}", fontsize=10); ax.grid(True, linestyle="--", alpha=0.5)
for ax in axes[-1]:
    ax.set_xlabel("T (years)")
plt.tight_layout(); plt.show()

### 11.4 Summary
All errors are implied volatility RMSE in vol points.

In [ ]:
validation_summary = pd.DataFrame({
    short[name]: {"parameters": cv.FAMILY[name],
                  "in-sample": cv.rmse(in_sample_errors[name]),
                  "leave-one-expiry-out": loo_table.loc["All", (short[name], "out")],
                  "strike holdout (test)": strike_table.loc[short[name], "test quotes"],
                  "noise : error vs original": noise_table.loc[short[name], "error vs original quotes"],
                  "noise : surface displacement": noise_table.loc[short[name], "surface displacement"]}
    for name in cv.FAMILY}).T
validation_summary.astype({"parameters": int}).round(3)

**Reading the results.**
- **Strikes: no overfitting.** On the strike holdout, the test error is 0.9 to 1.3 times the train error for every model,
  as for the constant model (1.14). The full models keep their advantage out of sample (0.136 - 0.185 vol pt against 0.462):
  three parameters per smile of 11 quotes do not fit the noise in the strike direction.
- **Maturities: real degradation.** Leaving one expiry out, the constant model goes from 0.42 to 0.52 vol pt, $\theta(t)$
  from 0.27 to 0.34, the full global model from 0.12 to 0.35 (three times its in-sample error) and the full bootstrap from
  0.16 to 0.39. Out of sample, the 26 extra parameters of the full global model buy nothing over $\theta(t)$ (0.35 vs 0.34):
  the parameters of one interval say little about the next one. With smoothness, the out-of-sample error is the lowest
  (0.245, half the one of the constant model): the penalty is what makes the time-dependent parameters interpolate.
  The 9-day expiry is the hardest to predict for the constant and full models (0.55 to 0.92 vol pt), since its skew is far
  from the next one; $\theta(t)$ predicts it to 0.19 vol pt, as its $\xi$ and $\rho$ are fitted on the whole surface.
- **Noise: the full models are unstable.** For the constant model and $\theta(t)$, a 0.1 vol pt noise moves the fitted surface
  by 0.017 and 0.040 vol pt, close to the least-squares reference $\sigma\sqrt{k/n}$ (0.019 and 0.033), and the parameters
  barely move. For the full models, the surface moves by 0.14 - 0.27 vol pt, i.e. 2.6 to 5 times the reference and more than
  the noise itself, and $\xi$ (standard deviation 0.36 - 0.66 in median, up to 1.9), $\rho$ (0.2 without penalty) and $\kappa$ (0.45 - 0.55 for the global runs)
  jump from one draw to the next. Yet their fit to the noisy quotes (0.17 - 0.26) is not below the reference: the models do not
  absorb the noise, the optimiser lands in different minima of a flat loss. This is non-identifiability, not classical overfitting.
  The smoothness penalty stabilises $\rho$ (median standard deviation 0.06) and halves the spread of $\xi$, but not the surface displacement.
- **Bootstrap**: the worst for stability (0.27 vol pt of displacement): each interval inherits the errors of the previous ones.

**Conclusion.** $\theta(t)$ is a safe improvement: 0.42 to 0.34 vol pt out of sample, as stable as the constant model. The full
time-dependent model fits better in-sample and between strikes, but its parameters are not identified by one day of vanillas:
it needs the smoothness penalty to interpolate in time, and even then its parameters and its surface depend on the noise of the
quotes. Stronger regularisation, or $\xi$ and $\rho$ on a few coarse maturity buckets instead of 13 intervals, would reduce the
number of effective parameters.